<a href="https://colab.research.google.com/github/Anant777-wq/flyrank-ml-internship-starter/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis:

One row represents one specific webpage (URL) during a single snapshot in time (a day).

Time Window:

The training data is strictly limited to April 2026 (2026-04). Any performance metrics occurring after April 30, 2026, are excluded from the training features to prevent data leakage.

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print("Loading warehouse data (Streaming Mode)...")

dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=hf_token, streaming=True)

sample_rows = list(dataset.take(5))
df_sample = pd.DataFrame(sample_rows)

print("\n✅ Columns available for our contract:")
print(df_sample.columns.tolist())

print("\n✅ Grain check (Sample rows):")
# Using the ACTUAL column names from the warehouse!
print(df_sample[['client_hash_id', 'report_date', 'content_hash_id', 'gsc_clicks']])
print("\nContract verified!")

Loading warehouse data (Streaming Mode)...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]


✅ Columns available for our contract:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

✅ Grain check (Sample rows):
            client_hash_id report_date           content_hash_id  gsc_clicks
0  client_9958f0a7ae1df715  2025-01-27  content_3b70a18ea133b2bb           0
1  client_9958f0a7ae1df715  2025-01-27  content_fe8e8155ce1d47a2           0
2  client_9958f0a7ae1df715  2025-01-27  content_b4462a1b90640058           0
3  client_9958f0a7ae1df715  2025-01-27  content_c899aef92518c714           0
4  client_9958f0a7ae

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

***Feature***: gsc_clicks, gsc_impressions, gsc_avg_position (from April).

***Why***: These are historical metrics fully knowable at the exact moment we make our decision on April 30th.

Label (Target): traffic_decline_next_30_days (A proxy we will calculate based on future data).

Why: This is what we are trying to predict.

***Context***: content_hash_id, report_date, client_hash_id.

***Why***: These don't help the model learn patterns; they just help us humans identify which row is which.

***Excluded***: gsc_clicks, gsc_impressions, gsc_avg_position from May and June.

***Why***: We must actively exclude future performance data. If the model sees May's metrics while making a prediction in April, it is cheating (Data Leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print("Loading data stream for verification...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=hf_token, streaming=True)

# Grab the first 500 rows instantly to verify our grain and column claims
print("Fetching 500 rows instantly...")
sample_rows = list(dataset.take(500))
df_verify = pd.DataFrame(sample_rows)

print("\n--- CONTRACT VERIFICATION ---")

print("\n1. Grain Check (Are these really daily rows per URL?):")
print(df_verify[['client_hash_id', 'content_hash_id', 'report_date']].head(3))

print("\n2. Missing Values Check (Do our features have nulls?):")
features = ['gsc_clicks', 'gsc_impressions', 'gsc_avg_position']
print(df_verify[features].isnull().sum())

print("\n3. Context Check (Are there multiple distinct URLs?):")
print(f"Unique URLs (content_hash_id) in this sample: {df_verify['content_hash_id'].nunique()}")

print("\nAll claims successfully backed by data!")

Loading data stream for verification...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Fetching 500 rows instantly...

--- CONTRACT VERIFICATION ---

1. Grain Check (Are these really daily rows per URL?):
            client_hash_id           content_hash_id report_date
0  client_9958f0a7ae1df715  content_3b70a18ea133b2bb  2025-01-27
1  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2  2025-01-27
2  client_9958f0a7ae1df715  content_b4462a1b90640058  2025-01-27

2. Missing Values Check (Do our features have nulls?):
gsc_clicks          0
gsc_impressions     0
gsc_avg_position    0
dtype: int64

3. Context Check (Are there multiple distinct URLs?):
Unique URLs (content_hash_id) in this sample: 336

All claims successfully backed by data!


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

***Unbalanced history***: Some clients have years of historical data, while newer clients might only have a few months.

***GSC-only early rows***: Older records (like early 2025) might only contain Google Search Console (GSC) data, missing Google Analytics (GA4) metrics entirely.

***Window overlaps***: When predicting a 30-day traffic decline for late April, the target window overlaps into May. We must ensure we don't accidentally use May's features to predict May's outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.